# 03 Cross-simulation test: TNG-trained model on Bolshoi

Applies the TNG300-3 dark-matter baseline to the Bolshoi dark-matter density with no retraining, and scores it against the Bolshoi T-web mask (same convention: sigma_eff = 1 h^-1 Mpc, threshold 0.65). Bolshoi was never seen in training, so the whole box counts as test data.

Two input normalizations are compared:
- **TNG statistics**: the normalization fit on the TNG training slab, exactly as in training.
- **Bolshoi statistics**: refit on the Bolshoi density itself. This uses no labels, so it is fair, and it removes a simple difference in the density distribution (Bolshoi has about 18 times more particles per cell than TNG300-3, so much less shot noise).

Runtime: **L4 or A100 GPU** with **High-RAM**. About 10 to 15 minutes.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
token = userdata.get('GITHUB_TOKEN')
!pip -q install --force-reinstall --no-deps "git+https://{token}@github.com/skumagai3/deepvoid.git"
del token

In [ ]:
import json
from pathlib import Path
import keras, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from deepvoid import Catalog
from deepvoid.evaluate import confusion_matrix, scores
from deepvoid.labels import CLASS_NAMES
from deepvoid.predict import predict_volume
from deepvoid.preprocess import Normalizer

run = Path('/content/drive/MyDrive/deepvoid_runs/tng_dm_baseline_v2')
cat = Catalog()
model = keras.models.load_model(run / 'best.keras')
tng_norm = Normalizer(**json.load(open(run / 'normalization.json'))['tng300-3/dm'])

dm = cat.load('bolshoi/dm').data
truth = cat.load('bolshoi/mask').data
print('Bolshoi', dm.shape, '| mask fractions', np.round(np.bincount(truth.ravel(), minlength=4) / truth.size, 3))

## Predict and score with both normalizations

In [ ]:
results, preds = {}, {}
for label, norm in [('TNG statistics', tng_norm), ('Bolshoi statistics', Normalizer.fit(dm))]:
    probs = predict_volume(lambda b: model.predict_on_batch(b), norm(dm), crop=128, margin=16, batch_size=4)
    preds[label] = probs.argmax(-1).astype(np.uint8)
    del probs
    cm = confusion_matrix(truth, preds[label])
    results[label] = scores(cm)
    print(label)
    print(pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES))

table = pd.DataFrame(results).loc[['mcc', 'void_f1', 'wall_f1', 'filament_f1', 'halo_f1', 'balanced_accuracy']]
table['TNG test slab'] = [0.941, 0.986, 0.943, 0.941, 0.911, 0.966]
json.dump(results, open(run / 'bolshoi_transfer_scores.json', 'w'), indent=2)
print(table.round(3))

In [ ]:
cmap = plt.matplotlib.colors.ListedColormap(['#2b6cb0', '#9ae6b4', '#f6ad55', '#c53030'])
k = dm.shape[0] // 2
panels = [('Bolshoi truth', truth[k])] + [(f'prediction, {l}', p[k]) for l, p in preds.items()]
fig, ax = plt.subplots(1, 3, figsize=(18, 6))
for a, (t, im) in zip(ax, panels):
    a.imshow(im.T, origin='lower', cmap=cmap, vmin=0, vmax=3, interpolation='nearest'); a.set_title(t); a.axis('off')
plt.tight_layout(); plt.show()

Paste the score table (and the figure, if you like) back to Claude.